In [ ]:
import os

from dotenv import load_dotenv
from typing import TypedDict
from langchain_google_genai import ChatGooleGenerativeAI
from langgraph.graph import StateGraph, START, END

load_dotenv(r"C:\Users\user\PythonProjecs\.env")

api_key = os.getenv("api_key")

print("API Key:", api_key)


class SubState(TypedDict):
    input_text: str
    translated_text: str

sub_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=api_key,
    temperature=0,
)


def translate_node(state: SubState):
    prompt = f"""
Translate the following English text into Hindi.

Text:
{state["input_text"]}
"""

    response = sub_llm.invoke(prompt)

    return {
        "translated_text": response.content
    }


sub_builder = StateGraph(SubState)

sub_builder.add_node("translate", translate_node)

sub_builder.add_edge(START, "translate")
sub_builder.add_edge("translate", END)

subgraph = sub_builder.compile()


class ParentState(TypedDict):
    question: str
    answer_english: str
    answer_hindi: str


parent_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=api_key,
    temperature=0,
)


def generate_english_node(state: ParentState):

    response = parent_llm.invoke(
        f"Answer this question in detail:\n\n{state['question']}"
    )

    return {
        "answer_english": response.content
    }


def invoke_subgraph_node(state: ParentState):

    result = subgraph.invoke(
        {
            "input_text": state["answer_english"]
        }
    )

    return {
        "answer_hindi": result["translated_text"]
    }


parent_builder = StateGraph(ParentState)

parent_builder.add_node(
    "generate_english",
    generate_english_node
)

parent_builder.add_node(
    "translate",
    invoke_subgraph_node
)

parent_builder.add_edge(
    START,
    "generate_english"
)

parent_builder.add_edge(
    "generate_english",
    "translate"
)

parent_builder.add_edge(
    "translate",
    END
)

parent_graph = parent_builder.compile()


result = parent_graph.invoke(
    {
        "question": "Why is the sky blue?"
    }
)

print("="*50)
print("Question:")
print(result["question"])

print("\nEnglish:")
print(result["answer_english"])

print("\nHindi:")
print(result["answer_hindi"])

API Key: AQ.Ab8RN6KpAyxFD2hwogCwLZ1lWbnGAUq-TRVrY04vcBI0ifX88g
Question:
Why is the sky blue?

English:
The sky appears blue due to a phenomenon called **Rayleigh Scattering**, which describes how light interacts with particles much smaller than its wavelength. Here's a detailed breakdown:

### 1. Sunlight is White Light

First, it's important to understand that the light from the sun, which appears white to us, is actually composed of all the colors of the rainbow (the visible spectrum: Red, Orange, Yellow, Green, Blue, Indigo, Violet). Each color corresponds to a different wavelength, with red having the longest wavelength and violet having the shortest.

### 2. Earth's Atmosphere

Our planet is surrounded by an atmosphere, which is a mixture of gases, primarily nitrogen (about 78%) and oxygen (about 21%). These gas molecules are incredibly tiny – much smaller than the wavelengths of visible light.

### 3. The Role of Rayleigh Scattering

When sunlight enters Earth's atmosphere, it e